# Stretch: more pandas, and more cleaning

For when the live exercises are done and there is time left. These get
harder as you go, so nobody is expected to finish them all. Stretches 1 to
4 build on the first half of the session, 5 to 7 on the cleaning.

Each one adds one idea, and the notebook tells you what it is when you get
there. Where a stretch has `assert` lines, uncomment them once your answer
exists: they are the real numbers, so they tell you when you are right.

Four tables are loaded: `plays` and `artists`, the clean CSVs, which
already carry the artist's name, genre and country on every play; `raw`,
the messy file exactly as `load_messy` returns it; and `cleaned`, the
result of `clean_plays(raw)`.

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd

here = Path.cwd()
while not (here / "data" / "messy" / "plays_messy.csv").exists() and here != here.parent:
    here = here.parent
os.chdir(here)

sys.path.insert(0, "sessions/session-06/solutions")
from cleaning import clean_plays, load_messy

plays = pd.read_csv("data/clean/plays.csv")
artists = pd.read_csv("data/clean/artists.csv")

raw = load_messy("data/messy/plays_messy.csv")
cleaned = clean_plays(raw)

print(plays.shape, artists.shape, raw.shape, cleaned.shape)

## Stretch 1: merging on a name

The plays CSV has `artist_name`, so you can merge `artists.csv` on by name.
Do it with `on="artist_name"`, and look at the columns that come back.

In [ ]:
# TODO: merged = plays.merge(artists, on="artist_name")
#       print(merged.columns.tolist())

`country` and `genre` are in **both** tables, so pandas keeps both and adds
`_x` (from the left) and `_y` (from the right) to tell them apart. Nothing
is wrong, but `genre_x` in a report is not a good look.

Do the merge again, this time selecting only the columns you need from
`artists` first, so there is nothing to clash. You want `artist_name` and
`formed_year`. Add `validate=` with the right relationship, and check the row
count is unchanged.

In [ ]:
# TODO


# assert len(merged) == len(plays)
# assert "formed_year" in merged.columns
# assert "genre_x" not in merged.columns

## Stretch 2: decades, and the plays that go missing

How many plays went to artists formed in each decade?

Make a `decade` column on `merged` from `formed_year`. Session 1's `//` does
it: `2014 // 10 * 10` is `2010`. It works on a whole column at once.

Then count the plays per decade with `groupby` and `.size()`.

In [ ]:
# TODO


# assert per_decade.loc[2010] == 1087

Now add the counts up. Is it 2,183?

In [ ]:
# TODO

**Think about it:** where did the missing plays go? There were no errors.
(Look back at question 3 of the session 5 homework. Which artists have no
`formed_year`, and what is their `decade`?)

*Your answer here.*

Find the option that makes `groupby` keep them. `help(merged.groupby)` lists
every option, or search for "pandas groupby dropna". Then prove the total is
right with an `assert`.

In [ ]:
# TODO

## Stretch 3: who is in one table and not the other?

In session 5 the never-played artists needed a `LEFT JOIN` and a
`WHERE ... IS NULL`. pandas has two ways.

**Way one:** `isin` and `~` (not). Which artist names in `artists` are not
in `plays["artist_name"]`?

In [ ]:
# TODO


# assert set(never) == {"Iron Fernway", "Gravel Choir"}

**Way two:** `merge` with `how="left"` and `indicator=True`. That adds a
column called `_merge` saying, for every row, whether it was found in
`"both"` tables or only the `"left_only"`. Merge `artists` with the
**distinct** artist names from `plays` (so there is no fan-out), and look at
`value_counts()` of `_merge`.

In [ ]:
# TODO

**Harder:** why "distinct artist names"? Try it with the whole of `plays`
instead, and compare the row counts.

In [ ]:
# TODO

## Stretch 4: the top artist in each genre

For every genre, which artist has the most minutes?

One way: group by **both** genre and artist to get the minutes per pair.
Sort that by minutes, biggest first. Then `groupby("genre").head(1)` keeps
the first row of each genre, and because you sorted first, the first row is
the biggest.

In [ ]:
# TODO


# assert top.set_index("genre").loc["Electronic", "artist_name"] == "Glass Tram"

Now change `head(1)` to `head(2)`, and look at how close second place is in
each genre. Is there a genre where you would not call the winner a winner?
Try the same thing by plays rather than minutes.

In [ ]:
# TODO

*Your answer here.*

## Stretch 5: a damage report

In notebook 04 you found the damage one column at a time. Write
`damage_report(df)`, which does it for every column at once and returns a
DataFrame with one row per column:

| column | blanks | spellings | after strip and lower |
|---|---|---|---|
| genre | 36 | 33 | 9 |

* `blanks`: how many values are the empty string `""`
* `spellings`: how many different values there are
* `after strip and lower`: how many different values are left after
  `.str.strip().str.lower()`

**New idea:** you build the report as a list of dictionaries, one per
column, and hand it to `pd.DataFrame`. Session 4 promised that a DataFrame
is a list of dicts with a faster engine; this is that promise, used
backwards.

Every column of `raw` holds text except `play_id` and `skipped`. Use
`.astype(str)` on each column first, so the same code works on both.

In [ ]:
# TODO: def damage_report(df):


# report = damage_report(raw)
# assert len(report) == 9
# assert report.set_index("column").loc["device", "spellings"] == 21
# assert report.set_index("column").loc["device", "after strip and lower"] == 6

Which column has the most spellings, and why is that one not damage at
all? Which column has a different number of `blanks` from what
`isna().sum()` reported on the default read in notebook 04, and why?

*Your answer here.*

## Stretch 6: what were the unknowns?

69 plays became `"unknown"`. The clean file knows what they really were.
Put the two side by side with `merge`, as in notebook 02.

**New idea:** `suffixes=("_ours", "_clean")` names the columns that appear
on both sides, and `validate="one_to_one"` makes pandas stop if a
`play_id` turns up twice on either side. Notebook 02 used `"many_to_one"`;
here each play exists once in each table.

In [ ]:
# TODO: merged = cleaned.merge(plays, on="play_id", ...)


# assert len(merged) == 2183
# unknowns = merged[merged["device_ours"] == "unknown"]
# assert len(unknowns) == 69

Count what the unknowns really were. Compare that with the share of each
device in the whole log (`value_counts(normalize=True)`).

Were the blanks spread across devices about evenly, or did one device lose
far more than its share? Why does that decide whether dropping them would
have been safe?

In [ ]:
# TODO

*Your answer here.*

## Stretch 7: guess the missing minutes, and get marked

94 plays have no minutes. Notebook 04 left them as `NaN`, and the total
came out at 8,608.0 instead of the clean file's 8,980.4.

Filling a gap with a best guess is called **imputing**. Try four ways and
compare the totals with the truth:

1. leave them missing
2. fill with the average of all plays
3. fill with the median of **that artist's** plays
4. fill with the median of that artist's plays **with the same `skipped`
   value** (skipped plays are short)

**New idea:** `cleaned.groupby("artist_name")["minutes_played"].transform("median")`
returns a column as long as `cleaned`, where every row holds its own artist's
median. That lines up with `minutes_played` row for row, so it can go
straight into `fillna`. `groupby` with a list of two columns works the same
way.

`imputed_total(minutes)` returns the total of a filled-in column. Then make
a small table: method, total, and the difference from 8,980.4.

In [ ]:
# TODO


# assert round(cleaned["minutes_played"].sum(), 1) == 8608.0

**Think about it:**

1. Method 3 is worse than method 2. Using more information made the guess
   worse. Why? (Look at `cleaned.loc[cleaned["minutes_played"].isna(), "skipped"].mean()`.)
2. Method 1 has the biggest error by far. Is it still the most honest?
   What would you write next to the total under each method?
3. In real work there is no clean file to mark your guess. How would you
   choose, then?

*Your answers here.*